# ICS40125 - Laboratorio N°04

Consolidación, limpieza y análisis del Índice de Libertad de Prensa.

In [ ]:
import numpy as np
import pandas as pd

archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')

## 1. Consolidación y limpieza de datos

In [ ]:
# a) unir ambos archivos por filas y pasar columnas a minuscula
df_anio = pd.concat([pd.read_csv(a) for a in archivos_anio], ignore_index=True)
df_anio.columns = df_anio.columns.str.lower()
df_anio.head()

In [ ]:
# b) revisar codigo iso asociado a dos paises distintos
conteo = df_codigos.groupby('codigo_iso')['pais'].nunique()
print("Codigos ISO con mas de un pais:")
print(conteo[conteo > 1])
print()
# ver los registros conflictivos
iso_malos = conteo[conteo > 1].index
print(df_codigos[df_codigos['codigo_iso'].isin(iso_malos)])

In [ ]:
# eliminar el registro inconsistente (nos quedamos con el primero de cada codigo)
df_codigos = df_codigos.drop_duplicates(subset='codigo_iso', keep='first')

In [ ]:
# c) combinar df_anio con df_codigos por codigo_iso (inner join)
df = pd.merge(df_anio, df_codigos, on='codigo_iso', how='inner')
df.head()

## 2. Exploración inicial del conjunto de datos

In [ ]:
# estructura
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])
print("Columnas:", list(df.columns))
print()
df.info()

In [ ]:
# resumen estadistico
df.describe()

In [ ]:
print("indice -> min:", df['indice'].min(), "| max:", df['indice'].max(), "| promedio:", round(df['indice'].mean(), 2))
print()
print("Menor indice (mayor libertad):")
print(df.loc[df['indice'].idxmin(), ['pais', 'anio', 'indice']])
print()
print("Mayor indice (menor libertad):")
print(df.loc[df['indice'].idxmax(), ['pais', 'anio', 'indice']])

In [ ]:
# datos faltantes
nulos = df.isnull().sum()
print("Nulos por columna:")
print(nulos)
print()
print("Proporcion de filas con algun nulo:", round(df.isnull().any(axis=1).mean(), 3))

In [ ]:
# unicidad y duplicados
print("Paises distintos:", df['pais'].nunique())
print("Anios distintos:", df['anio'].nunique())
print("Filas duplicadas:", df.duplicated().sum())

## 3. Comparación regional: países latinoamericanos

In [ ]:
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

df_america = df[df['codigo_iso'].isin(america)].copy()
df_america.head()

In [ ]:
# a) con ciclo for: pais con menor y mayor indice por anio
print("Extremos por anio (con for):")
for anio in sorted(df_america['anio'].unique()):
    sub = df_america[df_america['anio'] == anio]
    if len(sub) == 0:
        continue
    fila_min = sub.loc[sub['indice'].idxmin()]
    fila_max = sub.loc[sub['indice'].idxmax()]
    print(f"{anio}: mayor libertad -> {fila_min['pais']} ({fila_min['indice']}) | menor libertad -> {fila_max['pais']} ({fila_max['indice']})")

In [ ]:
# b) mismo resultado con groupby (vectorizado)
idx_min = df_america.groupby('anio')['indice'].idxmin()
idx_max = df_america.groupby('anio')['indice'].idxmax()

resumen = pd.DataFrame({
    'pais_mayor_libertad': df_america.loc[idx_min, 'pais'].values,
    'indice_min': df_america.loc[idx_min, 'indice'].values,
    'pais_menor_libertad': df_america.loc[idx_max, 'pais'].values,
    'indice_max': df_america.loc[idx_max, 'indice'].values
}, index=sorted(df_america['anio'].unique()))
resumen

## 4. Análisis anual del índice por país

In [ ]:
# tabla dinamica: filas=pais, columnas=anio, valores=indice maximo
tabla = pd.pivot_table(df, index='pais', columns='anio', values='indice',
                       aggfunc='max', fill_value=0)
tabla.head()

In [ ]:
# a) pais con mayor indice y menor (distinto de 0)
maximos = tabla.max(axis=1)
print("Pais con mayor indice:", maximos.idxmax(), "->", maximos.max())

sin_ceros = tabla.replace(0, np.nan).min(axis=1)
print("Pais con menor indice (>0):", sin_ceros.idxmin(), "->", sin_ceros.min())

In [ ]:
# b) anios con mayor y menor promedio
prom_anio = tabla.mean(axis=0)
print("Anio con mayor promedio:", prom_anio.idxmax())
print("Anio con menor promedio:", prom_anio.idxmin())

In [ ]:
# c) pais con mayor variabilidad
variabilidad = tabla.max(axis=1) - tabla.min(axis=1)
print("Pais con mayor variabilidad:", variabilidad.idxmax(), "->", variabilidad.max())

In [ ]:
# d) paises con indice constante
constantes = variabilidad[variabilidad == 0]
print("Paises con indice constante:", list(constantes.index))

In [ ]:
# e) paises sin datos (todo en 0)
sin_datos = tabla[(tabla == 0).all(axis=1)]
print("Paises sin datos:", list(sin_datos.index))
# Explicacion: quedaron en 0 porque no tenian registros de indice para ningun anio,
# probablemente por no aparecer en los archivos originales o perderse en el merge.